In [1]:

import env
from search import Stage1Search, SearchConfig
from cadgn import CADGNCore, TokenizerFamily
from models_config import models

search_config = SearchConfig(
    n_trials=100,
    search_epochs=15,
    full_epochs=150,
    pruning_warmup=10,
    n_startup_trials=5,
    results_dir="./search_results",
    storage="sqlite:///search.db",
    max_steps_train=550,
    max_steps_val=150,
)
do_models = [ "Qwen/Qwen3-4B", "meta-llama/Llama-3.2-3B", "Qwen/Qwen3-8B-FP8", "meta-llama/Llama-3.1-8B"]


In [2]:
from ds.cladder import CLadderDataset, load_cladder_v1_5, CLadderLoaderConfig

dsConfig = CLadderLoaderConfig(rung_filter=None, query_types=None, skip_unparseable=True)

train, vald = CLadderDataset.from_samples_split(
    samples=load_cladder_v1_5(dsConfig),
    val_size=0.2,
    stratify=True
)

Processing 10112 rows...
Loaded 8532 graphs, skipped 1580 unparseable rows.


In [3]:
search = Stage1Search(
    core_factory=lambda ap: CADGNCore(
        **ap.core_kwargs(),
    ),
    family_factory=lambda ap: [
        TokenizerFamily.from_pretrained(
            model_id= k,
            max_seq_len=128,
            torch_dtype=v['dtype'],
            **ap.family_kwargs(),
        )
        for k, v in models.items()
        if k in do_models
    ],
    train_dataset=train,
    val_dataset=vald,
    search_config=search_config,
)

In [4]:
study = search.run()

[I 2026-08-17 00:52:42,414] Using an existing study with name 'cadgn_stage1_search' instead of creating a new one.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

GPU: 0.72GB allocated / 0.72GB reserved


[transformers] FP8 quantized models is only supported on GPUs with compute capability >= 8.9 (e.g 4090/H100), actual = `8.6`. We will default to dequantizing the model to bf16. Feel free to use a different quantization method like bitsandbytes or torchao


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

...

[I 2026-08-17 23:19:13,282] Trial 150 pruned. 


GPU: 4.56GB allocated / 4.61GB reserved
Moving Qwen/Qwen3-4B's embed_layer from cuda to cuda
GPU: 3.61GB allocated / 3.63GB reserved
Moving Qwen/Qwen3-8B-FP8's embed_layer from cuda to cuda
GPU: 3.73GB allocated / 3.75GB reserved
Moving meta-llama/Llama-3.2-3B's embed_layer from cuda to cuda
GPU: 4.02GB allocated / 4.03GB reserved
Moving meta-llama/Llama-3.1-8B's embed_layer from cuda to cuda
GPU: 4.19GB allocated / 4.19GB reserved
Training Stage 1 for:
	Core: CADGNCore(
  (encoder): CADGNEncoder(
    (conv): CADGNConv(256, num_iters=4, epsilon=0.1402938137324587, base_gamma=0.020763793903883145)
    (dropout): Dropout(p=0.07131630818168577, inplace=False)
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): CADGNDecoder(
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (ffn): Sequential(
      (0): Linear(in_features=256, out_features=512, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.13768667587766653, inplace

[W 2026-08-17 23:26:46,216] Trial 151 failed with parameters: {'ca_dgn_dim': 256, 'max_layers': 5, 'num_iters': 4, 'epsilon': 0.1402938137324587, 'base_gamma': 0.020763793903883145, 'encoder_dropout': 0.07131630818168577, 'decoder_expansion': 2, 'decoder_dropout': 0.13768667587766653, 'head_encoder_dropout': 0.05558265636314816, 'head_decoder_dropout': 0.09491833769550252, 'lr': 0.0036331211076553464, 'weight_decay': 0.0006267818337843889, 'grad_accum_steps': 13} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/jerry/venvs/CausalADGN_venv/venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/mnt/d/Python Projects/CausalADGN/search.py", line 285, in _objective
    history = trainer.train_stage1(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/d/Python Projects/CausalADGN/CADGNTrainer.py", line 418, in train_stage1
    }
      

GPU: 4.56GB allocated / 4.61GB reserved


KeyboardInterrupt: 

In [ ]:
{print(k, v['dtype']) for k,v in models.items()}